## Setup — Install dependencies


In [1]:
pip install -r requirements.txt --trusted-host pypi.org --trusted-host files.pythonhosted.org --trusted-host pypi.python.org


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from snowflake.snowpark import Session
import json
import os
import re
import io
import uuid

import pandas as pd
from openpyxl import load_workbook

import cv2
import numpy as np
import matplotlib.pyplot as plt
import pytesseract
import pypdf
from PIL import Image, ImageFilter, ImageEnhance
from pdf2image import convert_from_path


In [3]:
BASE_DIR = os.getcwd()

excel_path      = os.path.join(BASE_DIR, "invoice_extract.xlsx")
input_directory = r'C:\Users\jakub.klosowski\email\Invoices\Xpd'

# Tesseract executable path (Windows)
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# Poppler path for PDF conversion (Windows)
POPPLER_PATH = r"C:\Program Files\poppler\poppler-25.12.0\Library\bin"

print("Excel Path:      ", excel_path)
print("Input Directory: ", input_directory)


Excel Path:       c:\Users\jakub.klosowski\email\FAP\xpd 2\invoice_extract.xlsx
Input Directory:  C:\Users\jakub.klosowski\email\Invoices\Xpd


In [4]:
try:
    version = pytesseract.get_tesseract_version()
    print(f"Tesseract version: {version}")
except Exception as e:
    print(f"Tesseract not found or misconfigured: {e}")


Tesseract version: 5.5.0.20241111


In [7]:
connection_parameters = {
    "user":          "JAKUB.KLOSOWSKI@AUTOLIV.COM",
    "authenticator": "externalbrowser",
    "account":       "VN02639-YR60386",
    "warehouse":     "DEV_PRINCIPAL_WH",
    "database":      "PROTO_DEVTEAM_DB",
    "schema":        "PUBLIC",
}

session = Session.builder.configs(connection_parameters).create()
print("Snowflake session created.")


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/f3603cca-3b48-4625-afd2-2eb0587bf1d6/saml2?SAMLRequest=nZJBb%2BIwEIX%2FSuQ9J3ESoMECKrYsWlbdLYK02vZmkgm1SOzgcQj0169DQOoe2kNvkfNmvjfzZnR7LAvnABqFkmMSeJQ4IFOVCbkdk8dk7sbEQcNlxgslYUxOgOR2MkJeFhWb1uZVrmBfAxrHNpLI2h9jUmvJFEeBTPISkJmUrae%2F71noUcYRQRuLI5eSDIVlvRpTMd9vmsZrIk%2FprR9SSn069K2qlXwj7xDV54xKK6NSVVxLjnamDxCBT3stwiosYXkp%2FC5kt4LPKJtOhOxnkizd5cM6Ic70Ot2dkliXoNegDyKFx9V9ZwCtg%2BfVgEbxwGvs3lyotarA42%2B1Bg%2BlavKC7yBVZVUb292zX34OmV%2BorbA7W8zGpNqJbBf9OuyPhyOVQ8zL7YEDJqf%2B23GTPycPL8Nm%2F5f3hz92yTzrpcR5uiYctgkvEGtYyDZXY59oOHBp5AZxElAWxax%2F4%2FUi%2BkKcmfUnJDfnyqv5sw%2BvFKlWqHKjZCEkdC4jO1aacjfa9GK3Nwj7Ls%2Bz0A1hQ%2FvxzSYPsoHfpheS7oLY2YiefHUvI%2F99l8tR%2FrE5LWZLVYj05MyVLrn5OMbAC84vInPzs5RByUUxzTINiDbOolDNnQZu7O0bXQPxJx31%2F%2Buf%2FAM%3D&RelayState=ver%3A3-hint%3A3302906936078-ETMsDgAAAZ0AhvOWABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAED9OoBeu9IPgF9VA%2FnoMLF

## Cortex LLM — Connectivity Test


In [8]:
# Quick sanity check — verify Cortex LLM is reachable
result = session.sql("""
    SELECT snowflake.cortex.complete('llama3.1-70b', 'Say hello in one word.')
""").collect()
print(result[0][0])


Hello!


In [29]:
def ask_cortex_invoice_data(session, extracted_text):
    try:
        prompt = f"""
Return only the JSON.

The following is an OCR extract of an XPD Global / Europartners Mexico invoice. The document has TWO pages separated by markers:
  === OCR: Full Page 1 (Real Positions) ===  ... === End of Page 1 ===
  === OCR: Full Page 2 (Real Positions) ===  ... === End of Page 2 ===

Page 1 and Page 2 contain the same invoice content. Extract all fields from Page 1. Use Page 2 only if a field cannot be found on Page 1.

Your task is to extract the following fields with ABSOLUTE PRECISION.
Each field MUST be extracted exactly as it appears in the OCR text — do not infer, calculate, or paraphrase.

=== FIELD DEFINITIONS AND EXACT LOCATIONS ===

1. vendor
   This is a fixed constant value — always return exactly: "XPD Global".

2. invoice_number
   SOURCE: Page 1 only.
   In the upper-right area of Page 1 there is a row containing the label "Serie y folio interno".
   The invoice number is the value to the RIGHT of that label on the same line.
   It is an alphanumeric code starting with "FCUS-" followed by digits — e.g. "FCUS-620207".
   IMPORTANT: Extract the COMPLETE value including ALL digits after "FCUS-". Do NOT truncate or drop any digits.

3. bill_to
   The "Bill To" recipient name and address.
   Always set bill_to to 'Autoliv ASP, Inc.     3350 Airport Road     Ogden, UT 84405   USA'. Do not extract from the document.

4. invoice_date
   SOURCE: Page 1 only.
   Found in the RIGHT column of Page 1, on the line containing the label "Fecha/Date:".
   Extract the date value to the RIGHT of that label.
   Example: on the line "Fecha/Date: 1/20/2026", extract "1/20/2026".

5. ship_date
   SOURCE: Page 1 only.
   Found in a section labeled "ETD/ETA:" in the middle area of Page 1.
   Two dates appear on the same line below or next to "ETD/ETA:" — the FIRST date is the ETD (ship date).
   Example: on the line "1/15/2026               1/15/2026", extract "1/15/2026".

6. shipper
   SOURCE: Page 1 only.
   The label "Embarcador/Shipper:" appears on the LEFT side of Page 1, roughly in the upper-middle area.
   The shipper block consists of lines BELOW that label, on the LEFT side of the document.
   Collect lines until you reach the "Consignatario/Consignee:" label.
   Combine all collected lines into one string, parts separated by ", ".

7. consignee
   SOURCE: Page 1 only.
   The label "Consignatario/Consignee:" (or "Consianatario/Consianee:") appears on the LEFT side of Page 1, below the shipper block.
   Collect lines until you reach the "Linea/Line:" label or the next section header.
   Combine all collected lines into one string, parts separated by ", ".

8. tms_number, ptn_number, autoliv_person_concerned
   SOURCE: Page 1 only.
   Found under the label "Fact. Comercial:/Comercial INV:".
   Collect LEFT column lines only. Stop at "Corresponsal:/Agent:".
   Do NOT use content from the "Marcas y Nos:/Marks" section.
   Combine all collected lines into one string, parts separated by " // ".

   The collected text will match ONE of two patterns — identify which and extract accordingly:

   PATTERN A — TMS reference (when any fragment equals "TMS" or starts with "tms"):
     Raw format: "TMS // load <number> // <First Last> // <company>"
     Example:    "TMS // load 881011 // Melissa Sanchez // MQL - AOA"
     → tms_number               = "load 881011"
     → ptn_number               = "N/A"
     → autoliv_person_concerned = "Melissa Sanchez"

   PATTERN B — PTN reference (when any fragment starts with "PTN:"):
     Raw format: "<code> // PTN: <ptn-code> // <First Last> // <company>"
     Example:    "ASP // PTN: 20CD-JG70-1GQYL // Hugo Acosta // LEMTECH. China"
     → tms_number               = "N/A"
     → ptn_number               = "20CD-JG70-1GQYL"
     → autoliv_person_concerned = "Hugo Acosta"

   RULES:
   - The person's name is always the fragment immediately AFTER the TMS or PTN identifier.
   - It consists of two words (first name + last name). Do NOT include the company name.
   - If TMS is present → populate tms_number, set ptn_number to "N/A".
   - If PTN is present → populate ptn_number, set tms_number to "N/A".
   - If neither pattern is found → set all three fields to "N/A".

9. weight_value and weight_unit
    SOURCE: Page 1 only.
    Found in a row labeled "Peso Bruto:/Gross weigth" in the lower-middle area of Page 1.
    Extract the numeric value and the unit separately.
    Example: weight_value = 4.314, weight_unit = "KG"

10. currency
    SOURCE: Page 1 only.
    Extract the 3-letter currency code from the first line item below "Impuesto:".
    All line items share the same currency — extract it once.
    Example: "USD"

11. line_items
    SOURCE: Page 1 only.
    Located BELOW the "Impuesto:" header. Each row begins with a numeric code (e.g. "81141601").
    For EACH row extract: code, description, quantity, unit_price, tax_type, currency, amount.
    Return as a JSON array of objects — one object per line item.
    Stop when you reach a line starting with "(" followed by an amount written in words.
    If no line items are found, return [].
    Example:
    [
      {{"code": "81141601", "description": "Air Freight", "quantity": "0", "unit_price": "0.00", "tax_type": "E48", "currency": "USD", "amount": "17,310.00"}},
      {{"code": "81141601", "description": "Destination Charges", "quantity": "0", "unit_price": "0.00", "tax_type": "E48", "currency": "USD", "amount": "1,500.00"}},
      {{"code": "81141601", "description": "Origin Charges", "quantity": "0", "unit_price": "0.00", "tax_type": "E48", "currency": "USD", "amount": "2,920.00"}}
    ]

12. amount
    SOURCE: Page 1 only.
    On the line containing the label "TOTAL", extract the numeric value to the RIGHT of "TOTAL".
    Do NOT extract "PARCIAL" or "I.V.A." — only the final "TOTAL" line.
    Example: "21,730.00"

13. mawb
    SOURCE: Page 1 only.
    Found in the RIGHT area of Page 1, on the row containing the label "B/L:MAWB:".
    The value appears on the line IMMEDIATELY BELOW that label, at the column position of "B/L:MAWB:".
    It is a numeric shipment reference code, e.g. "020-01408691".
    If not found, return "N/A".

14. hawb
    SOURCE: Page 1 only.
    Found in the FAR RIGHT area of Page 1, on the row containing the label "FBL:/HAW:".
    The value appears on the line IMMEDIATELY BELOW that label, at the column position of "FBL:/HAW:".
    It is an alphanumeric house airway bill code, e.g. "FRA-00009465".
    If not found, return "N/A".

15. bultos
    SOURCE: Page 1 only.
    Found in the lower area of Page 1, on the row containing the label "Bultos:".
    The value appears on the line with the column position of "Bultos:".
    It is a numeric value, e.g. "10".
    If not found, return "N/A".

=== RULES ===
- If a field cannot be found, set its value to "N/A" (or [] for line_items).
- Do NOT infer, estimate, or calculate — extract only what is literally present in the OCR.
- Do NOT include any explanation, commentary, or markdown — return ONLY the raw JSON.
- Return a single JSON array with exactly one object.

[
  {{
    "vendor": ...,
    "transport_mode": ...,
    "invoice_number": ...,
    "bill_to": ...,
    "invoice_date": ...,
    "ship_date": ...,
    "shipper": ...,
    "consignee": ...,
    "tms_number": ...,
    "ptn_number": ...,
    "autoliv_person_concerned": ...,
    "weight_value": ...,
    "weight_unit": ...,
    "currency": ...,
    "line_items": [...],
    "amount": ...,
    "mawb": ...,
    "hawb": ...,
    "bultos": ...
  }}
 ]

Document text:
\"\"\"
{extracted_text}
\"\"\"
"""
        escaped_prompt = prompt.replace("'", "\\'")
        sql = f"""
            SELECT snowflake.cortex.complete(
                'llama3.1-70b',
                $$ {escaped_prompt} $$
            )
        """
        result = session.sql(sql).collect()
        summary_text = result[0][0]
        return summary_text.strip(), None
    except Exception as e:
        return None, f"Failed to run Cortex Completion: {e}"


## Excel Writer


In [ ]:
def insert_invoice_data(json_data, excel_path):
    """Appends a single invoice record to the Excel file (creates file if it doesn't exist).
    line_items (list of objects) is serialised to a JSON string in the LINE_ITEMS column.
    """
    if isinstance(json_data, str):
        json_data = json_data.strip("`\n ")
        json_data = json.loads(json_data)

    line_items = json_data.get("line_items", [])
    line_items_str = json.dumps(line_items, ensure_ascii=False) if line_items else ""

    row_data = {
        "FILENAME":                  json_data.get("filename", ""),
        "VENDOR":                    json_data.get("vendor", ""),
        "INVOICE_NUMBER":            json_data.get("invoice_number", ""),
        "SHIPPER":                   json_data.get("shipper", ""),
        "CONSIGNEE":                 json_data.get("consignee", ""),
        "SHIP_DATE":                 json_data.get("ship_date", ""),
        "TMS_NUMBER":                json_data.get("tms_number", ""),
        "PTN_NUMBER":                json_data.get("ptn_number", ""),
        "AUTOLIV_PERSON_CONCERNED":  json_data.get("autoliv_person_concerned", ""),
        "MAWB":                      json_data.get("mawb", ""),
        "HAWB":                      json_data.get("hawb", ""),
        "WEIGHT_VALUE":              json_data.get("weight_value", ""),
        "WEIGHT_UNIT":               json_data.get("weight_unit", ""),
        "LINE_ITEMS":                line_items_str,
        "AMOUNT":                    json_data.get("amount", ""),
    }

    df_new = pd.DataFrame([row_data])

    os.makedirs(os.path.dirname(excel_path), exist_ok=True)

    if os.path.exists(excel_path):
        book = load_workbook(excel_path)
        sheet = book["Sheet1"]
        startrow = sheet.max_row

        with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
            df_new.to_excel(writer, sheet_name="Sheet1", startrow=startrow, index=False, header=False)
    else:
        df_new.to_excel(excel_path, sheet_name="Sheet1", index=False)

    print(f"Inserted 1 row successfully into {excel_path}")


## Image Preprocessing


In [17]:
def deskew_image_pil(pil_img, show=False):
    """
    Deskew image using horizontal projection profile.
    Scans angles from -15° to +15° and picks the one that maximises row-sum variance
    (well-aligned text produces sharp peaks in the row projection).
    """
    img = np.array(pil_img.convert("L"))

    # Binarise to isolate text pixels
    img_bin = cv2.adaptiveThreshold(
        img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 15, 8
    )

    h, w = img_bin.shape
    center = (w // 2, h // 2)

    best_angle, best_score = 0.0, -1.0
    for angle in np.arange(-15, 15.1, 0.5):
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        rotated = cv2.warpAffine(
            img_bin, M, (w, h), flags=cv2.INTER_NEAREST,
            borderMode=cv2.BORDER_CONSTANT, borderValue=0,
        )
        score = float(np.var(np.sum(rotated, axis=1)))
        if score > best_score:
            best_score, best_angle = score, angle

    print(f"Detected skew angle: {best_angle:.1f}°")

    M = cv2.getRotationMatrix2D(center, best_angle, 1.0)
    rotated_img = cv2.warpAffine(
        img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE
    )
    pil_rotated = Image.fromarray(rotated_img)

    if show:
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))
        axes[0].imshow(pil_img, cmap="gray" if pil_img.mode == "L" else None)
        axes[0].set_title("Before deskew")
        axes[0].axis("off")
        axes[1].imshow(pil_rotated, cmap="gray")
        axes[1].set_title(f"After deskew ({best_angle:.1f}°)")
        axes[1].axis("off")
        plt.tight_layout()
        plt.show()

    return pil_rotated


In [22]:
def preprocess_image_pil(img):
    """
    Prepare image for OCR:
      1. Convert to greyscale
      2. Enhance contrast (factor 1.8)
      3. Apply adaptive binarisation (Gaussian, block 21, C 10)
    """
    if img.mode != "L":
        img = img.convert("L")

    img = ImageEnhance.Contrast(img).enhance(1.8)

        # 5. Skalowanie (medium: 1.5)
    new_width = int(img.width * 1.5)
    new_height = int(img.height * 1.5)
    img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)

    img_bin = cv2.adaptiveThreshold(
        np.array(img), 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY,
        21, 10,
    )
    return Image.fromarray(img_bin)


In [30]:
def _create_positioned_line(line_items, chars_per_pixel=17):
    """Reconstruct a single text line preserving original X-positions as spaces."""
    line_items.sort(key=lambda x: x["x"])
    if not line_items:
        return ""

    first = line_items[0]
    result = " " * max(0, first["x"] // chars_per_pixel) + first["text"]
    current_x = first["x"] + len(first["text"]) * chars_per_pixel

    for item in line_items[1:]:
        spaces = max(1, (item["x"] - current_x) // chars_per_pixel)
        result += " " * spaces + item["text"]
        current_x = item["x"] + len(item["text"]) * chars_per_pixel

    return result


def extract_with_real_positions(image_input, chars_per_pixel=17):
    """
    Run Tesseract OCR and reconstruct text with spatial layout preserved.
    Words are placed at their real pixel X positions so columns remain aligned.

    Args:
        image_input: file path (str) or PIL Image
        chars_per_pixel: pixel-to-character ratio controlling column spacing
    """
    image = Image.open(image_input) if isinstance(image_input, str) else image_input

    data = pytesseract.image_to_data(
        image,
        output_type=pytesseract.Output.DICT,
        config="--oem 1 --psm 11 --dpi 300",
    )

    # Keep all non-empty tokens (confidence filter disabled intentionally)
    tokens = [
        {"text": data["text"][i], "x": data["left"][i], "y": data["top"][i],
         "width": data["width"][i], "height": data["height"][i]}
        for i in range(len(data["text"]))
        if data["text"][i].strip()
    ]

    # Sort top-to-bottom, left-to-right and group into lines (±17 px tolerance)
    tokens.sort(key=lambda t: (t["y"], t["x"]))
    lines, current_line, current_y = [], [], None

    for token in tokens:
        if current_y is None or abs(token["y"] - current_y) < 17:
            current_line.append(token)
            current_y = token["y"]
        else:
            lines.append(_create_positioned_line(current_line, chars_per_pixel))
            current_line = [token]
            current_y = token["y"]

    if current_line:
        lines.append(_create_positioned_line(current_line, chars_per_pixel))

    return "\n".join(lines)


In [31]:
def _pdf_to_images(pdf_path, max_pages=None):
    """Convert PDF pages to PIL images using Poppler."""
    kwargs = {"dpi": 300, "poppler_path": POPPLER_PATH}
    if max_pages:
        kwargs.update({"first_page": 1, "last_page": max_pages})
    return convert_from_path(pdf_path, **kwargs)


def pdf_to_ocr_text(pdf_path, max_pages=None):
    """
    Convert all pages of a PDF to position-aware OCR text.
    Each page is wrapped with markers:
        === OCR: Full Page N (Real Positions) === ... === End of Page N ===
    """
    print(f"OCR: {os.path.basename(pdf_path)}"
          + (f" (first {max_pages} page(s))" if max_pages else ""))

    images = _pdf_to_images(pdf_path, max_pages)
    if not images:
        print("  ERROR: could not convert PDF to images.")
        return None

    pages = []
    for i, image in enumerate(images, 1):
        try:
            preprocessed = preprocess_image_pil(image)
        except Exception as exc:
            print(f"  WARN: preprocessing failed on page {i}: {exc} — using greyscale.")
            preprocessed = image.convert("L")

        page_text = extract_with_real_positions(preprocessed)
        pages.append(
            f"\n=== OCR: Full Page {i} (Real Positions) ===\n"
            f"{page_text.strip()}\n"
            f"=== End of Page {i} ==="
        )

    return "\n".join(pages).strip()


def get_numbered_ocr_text(pdf_path, max_pages=None):
    """Return OCR text with line numbers (useful for debugging LLM extraction)."""
    raw = pdf_to_ocr_text(pdf_path, max_pages)
    if not raw:
        return "ERROR: OCR failed."

    lines = raw.split("\n")
    numbered = [
        f"{i:3d}: {line}" if line.strip() else f"{i:3d}: [empty]"
        for i, line in enumerate(lines, 1)
    ]
    numbered.append(f"\nTotal lines: {len(lines)}")
    return "\n".join(numbered)


def batch_ocr_to_txt(input_dir, output_dir, max_pages=None):
    """Process all PDFs in input_dir, write one .txt OCR file per PDF to output_dir."""
    os.makedirs(output_dir, exist_ok=True)
    pdf_files = [f for f in os.listdir(input_dir) if f.lower().endswith(".pdf")]
    if not pdf_files:
        print(f"No PDF files found in: {input_dir}")
        return

    for filename in pdf_files:
        txt_path = os.path.join(output_dir, os.path.splitext(filename)[0] + ".txt")
        text = get_numbered_ocr_text(os.path.join(input_dir, filename), max_pages)
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(text)
        print(f"  Saved: {txt_path}")


In [25]:
print(batch_ocr_to_txt(input_directory, os.path.join(BASE_DIR, "ocr_output"), max_pages=1))

OCR: ARInvoice-FCUS-624688.pdf (first 1 page(s))
  Saved: c:\Users\jakub.klosowski\email\FAP\xpd 2\ocr_output\ARInvoice-FCUS-624688.txt
OCR: ARInvoice-FCUS-626918.pdf (first 1 page(s))
  Saved: c:\Users\jakub.klosowski\email\FAP\xpd 2\ocr_output\ARInvoice-FCUS-626918.txt
OCR: ARInvoice-FCUS-626985.pdf (first 1 page(s))
  Saved: c:\Users\jakub.klosowski\email\FAP\xpd 2\ocr_output\ARInvoice-FCUS-626985.txt
OCR: ARInvoice-FCUS-627285.pdf (first 1 page(s))
  Saved: c:\Users\jakub.klosowski\email\FAP\xpd 2\ocr_output\ARInvoice-FCUS-627285.txt
OCR: ARInvoice-FCUS-627506.pdf (first 1 page(s))
  Saved: c:\Users\jakub.klosowski\email\FAP\xpd 2\ocr_output\ARInvoice-FCUS-627506.txt
OCR: Europartners XPD Vendor Invoice Data Points.pdf (first 1 page(s))
  Saved: c:\Users\jakub.klosowski\email\FAP\xpd 2\ocr_output\Europartners XPD Vendor Invoice Data Points.txt
None


In [32]:
input_directory = r"C:\Users\jakub.klosowski\email\aib\Invoices\xpd 2"


def process_directory_ocr_stage1(input_dir, max_pages=2):
    """
    Stage 1 pipeline: for each PDF in input_dir
      1. Run OCR  →  position-aware text
      2. Send text to Cortex LLM  →  structured JSON
      3. Append result to Excel
      4. Collect all records for Snowflake staging insert
    """
    print(f"Scanning: {input_dir}")
    print("=" * 60)

    raw_invoices_data = []
    local_excel_path  = os.path.join(input_dir, "invoice_extract.xlsx")
    total_saved       = 0

    for filename in sorted(os.listdir(input_dir)):
        if not filename.lower().endswith(".pdf"):
            continue

        print(f"\nProcessing: {filename}")
        extracted_text = pdf_to_ocr_text(
            os.path.join(input_dir, filename), max_pages=max_pages
        )

        if not extracted_text:
            print(f"  ERROR: OCR returned nothing for {filename}")
            continue

        structured_output, error = ask_cortex_invoice_data(session, extracted_text)
        if error:
            print(f"  ERROR (Cortex): {error}")
            continue

        try:
            invoices = json.loads(structured_output)
        except json.JSONDecodeError as e:
            print(f"  ERROR (JSON parse): {e}\n  Raw response: {structured_output}")
            continue

        for invoice in invoices:
            invoice["filename"] = filename
            try:
                insert_invoice_data(invoice, local_excel_path)
                total_saved += 1
                print(f"  OK  {invoice.get('invoice_number', 'N/A')} -> Excel")
            except Exception as exc:
                print(f"  ERROR (Excel): {exc}")
        print("✅ Structured Data extracted:")
        print(json.dumps(invoices, indent=2, ensure_ascii=False))

        raw_invoices_data.extend(invoices)

    print("=" * 60)
    print(f"Invoices extracted : {len(raw_invoices_data)}")
    print(f"Saved to Excel     : {total_saved}")
    return raw_invoices_data


raw_invoices_data = process_directory_ocr_stage1(input_directory, max_pages=2)


Scanning: C:\Users\jakub.klosowski\email\aib\Invoices\xpd 2

Processing: ARInvoice-FCUS-624688.pdf
OCR: ARInvoice-FCUS-624688.pdf (first 2 page(s))
  ERROR (Excel): name 'insert_invoice_data' is not defined
✅ Structured Data extracted:
[
  {
    "vendor": "XPD Global",
    "transport_mode": "N/A",
    "invoice_number": "FCUS-624688",
    "bill_to": "Autoliv ASP, Inc.     3350 Airport Road     Ogden, UT 84405   USA",
    "invoice_date": "1/20/2026",
    "ship_date": "1/15/2026",
    "shipper": "EICHSFELDER  SCHRAUBENWERK     GMBH, Rengelréder Weg  13 Heilbad, Heiligenstadt Thiiringen 37308 Germany  DE811210116",
    "consignee": "AUTOLIV  AST DEPOT  / CASAS INT 9355 Airway Road. suite 4 San Diego, California 92154 United States",
    "tms_number": "N/A",
    "ptn_number": "2091-F5F4-1GQYL",
    "autoliv_person_concerned": "Sirenia Tirado",
    "weight_value": 4.314,
    "weight_unit": "KG",
    "currency": "USD",
    "line_items": [
      {
        "code": "81141601",
        "descripti

In [35]:
import uuid
import json

def convert_mmddyyyy_to_ddmmyyyy(date_str):
    """Convert MM/DD/YYYY to DD/MM/YYYY. Returns original if not valid."""
    import re
    match = re.match(r'^(\d{1,2})/(\d{1,2})/(\d{4})$', str(date_str).strip())
    if not match:
        return date_str
    mm, dd, yyyy = match.groups()
    return f"{dd.zfill(2)}/{mm.zfill(2)}/{yyyy}"

# Przykład: konwersja dat w rekordach
for invoice in raw_invoices_data:
    for key in ['invoice_date', 'ship_date']:
        if key in invoice:
            invoice[key] = convert_mmddyyyy_to_ddmmyyyy(invoice[key])

def _esc(val):
    if val is None or str(val).strip() in ('', 'N/A', 'n/a'):
        return 'NULL'
    return "'" + str(val).replace("'", "''") + "'"


def _parse_amount(s):
    try:
        return float(str(s).replace(',', ''))
    except Exception:
        return None


def _parse_weight_value(s):
    try:
        return float(str(s).strip().replace(',', '.'))
    except Exception:
        return None


def insert_invoices_to_staging(
    session,
    raw_invoices_data,
    header_table='STG_INVOICE_HEADER',
    charge_table='STG_INVOICE_CHARGE_LINES',
    ref_table='STG_INVOICE_REFERENCES',
):
    if not raw_invoices_data:
        print("No data to insert.")
        return

    existing = {
        r[0] for r in session.sql(f"SELECT INVOICE_ID FROM {header_table}").collect() if r[0]
    }
    print(f"Already in {header_table}: {len(existing)} invoice(s)")

    inserted_h = inserted_c = inserted_r = 0

    for inv in raw_invoices_data:
        invoice_id = str(inv.get('invoice_number') or '').strip()
        if not invoice_id or invoice_id == 'N/A':
            print(f"  SKIP (no invoice_number): {inv.get('filename', '?')}")
            continue
        if invoice_id in existing:
            print(f"  SKIP (duplicate): {invoice_id}")
            continue

        weight_val  = _parse_weight_value(inv.get('weight_value'))
        amount_val  = _parse_amount(inv.get('amount'))
        ocr_json    = json.dumps(inv, ensure_ascii=False)
        ocr_escaped = "'" + ocr_json.replace("'", "''") + "'"

        # ── STG_INVOICE_HEADER ─────────────────────────────────────────────
        session.sql(f"""
            INSERT INTO {header_table} (
                INVOICE_ID, INVOICE_DATE, DUE_DATE,
                SHIP_DATE, DELIVERY_DATE, DELIVERY_TIME, DELIVERY_TIMEZONE,
                CARRIER_NAME_RAW, BILL_TO_RAW,
                INVOICE_TOTAL, CURRENCY,
                TRANSPORT_MODE,
                SHIP_FROM, SHIP_TO,
                WEIGHT_VALUE, WEIGHT_UNIT,
                SOURCE_CARRIER, OCR_RAW_PAYLOAD,
                LOAD_TIMESTAMP, PROCESSING_STATUS
            )
            SELECT
                {_esc(invoice_id)},
                TRY_TO_DATE({_esc(inv.get('invoice_date'))}, 'DD/MM/YYYY'),
                NULL,
                TRY_TO_DATE({_esc(inv.get('ship_date'))}, 'DD/MM/YYYY'),
                NULL,
                NULL,
                NULL,
                {_esc(inv.get('vendor'))},
                {_esc(inv.get('bill_to'))},
                {amount_val if amount_val is not None else 'NULL'},
                {_esc(inv.get('currency'))},
                'Air',
                {_esc(inv.get('shipper'))},
                {_esc(inv.get('consignee'))},
                {weight_val if weight_val is not None else 'NULL'},
                {_esc(inv.get('weight_unit'))},
                {_esc(inv.get('vendor'))},
                PARSE_JSON({ocr_escaped}),
                CURRENT_TIMESTAMP,
                'NEW'
        """).collect()
        inserted_h += 1

        # ── STG_INVOICE_CHARGE_LINES ──────────────────────────────────────
        line_items = inv.get('line_items') or []
        if isinstance(line_items, str):
            try:
                line_items = json.loads(line_items)
            except Exception:
                line_items = []

        for seq, item in enumerate(line_items, 1):
            if not isinstance(item, dict):
                continue
            charge_amount = _parse_amount(item.get('amount'))
            session.sql(f"""
                INSERT INTO {charge_table} (
                    CHARGE_LINE_ID, INVOICE_ID, SOURCE_CARRIER,
                    CHARGE_SEQUENCE, CHARGE_DESCRIPTION_RAW,
                    CHARGE_QUANTITY, CHARGE_UNIT_PRICE,
                    CHARGE_AMOUNT, CURRENCY
                ) VALUES (
                    {_esc(item.get('code'))},
                    {_esc(invoice_id)},
                    {_esc(inv.get('vendor'))},
                    {seq},
                    {_esc(item.get('description'))},
                    {_parse_amount(item.get('quantity')) if _parse_amount(item.get('quantity')) is not None else 'NULL'},
                    {_parse_amount(item.get('unit_price')) if _parse_amount(item.get('unit_price')) is not None else 'NULL'},
                    {charge_amount if charge_amount is not None else 'NULL'},
                    {_esc(item.get('currency'))}
                )
            """).collect()
            inserted_c += 1

        # ── STG_INVOICE_REFERENCES ────────────────────────────────────────
        references = [
            ('TMS', inv.get('tms_number')),
            ('PTN',        inv.get('ptn_number')),
            ('AUTOLIV_PERSON_CONCERNED', inv.get('autoliv_person_concerned')),
            ('MAWB',      inv.get('mawb')),
            ('HAWB',      inv.get('hawb')),
        ]

        for ref_type, ref_value in references:
            session.sql(f"""
                INSERT INTO {ref_table} (
                    REFERENCE_ID, INVOICE_ID, SOURCE_CARRIER,
                    REFERENCE_TYPE, REFERENCE_VALUE
                ) VALUES (
                    {_esc(str(uuid.uuid4()))},
                    {_esc(invoice_id)},
                    {_esc(inv.get('vendor'))},
                    {_esc(ref_type)},
                    {_esc(ref_value)}
                )
            """).collect()
            inserted_r += 1

        existing.add(invoice_id)
        print(f"  OK  {invoice_id}")

    print(f"\nSTG_INVOICE_HEADER:       {inserted_h} row(s)")
    print(f"STG_INVOICE_CHARGE_LINES: {inserted_c} row(s)")
    print(f"STG_INVOICE_REFERENCES:   {inserted_r} row(s)")


# ── Execute ───────────────────────────────────────────────────────────────────
session.sql("USE DATABASE DEV_SCM_DB").collect()
session.sql("USE SCHEMA OUT_TMS").collect()

insert_invoices_to_staging(session, raw_invoices_data)


Already in STG_INVOICE_HEADER: 14 invoice(s)
  OK  FCUS-624688
  OK  FCUS-626918
  OK  FCUS-626985
  OK  FCUS-627285
  OK  FCUS-627506
  OK  FCUS-620207

STG_INVOICE_HEADER:       6 row(s)
STG_INVOICE_CHARGE_LINES: 15 row(s)
STG_INVOICE_REFERENCES:   30 row(s)
